In [1]:
import numpy as np
import pandas as pd

In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [3]:
import re

def text_clean(text):
  lower_text = text.lower()
  simple_text = re.sub(r'[^a-z0-9\s]', '', lower_text)
  cleaned_text = re.sub(r'\s+', ' ', simple_text)
  return cleaned_text

In [4]:
#applying function to all the columns in both train and test
cols = ['prompt' , 'A', 'B' , 'C' , 'D' , 'E']

for col in cols:
    train[col +'_clean'] = train[col].apply(text_clean)
    test[col +'_clean'] = test[col].apply(text_clean)

In [5]:
train.head(2)

,id,prompt,A,B,C,D,E,answer,prompt_clean,A_clean,B_clean,C_clean,D_clean,E_clean
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer what is martin h...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is acceleratorbased lightion fusion,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...


In [6]:
def map_at3(prediction,correct):  #function to calculate MAP@3 score
  score = 0
  for pred, x in zip(prediction, correct):
    if x in pred:
      k = pred.index(x) + 1
      score += 1/k
  return round(score/len(correct) ,4)

In [7]:
x = train.copy()
x = x.drop_duplicates(subset = 'prompt_clean').reset_index(drop = True)
x.shape

(1758, 14)

In [8]:
from sklearn.model_selection import train_test_split

x_train, x_test = train_test_split(x, test_size=0.15, random_state=42 , stratify = x['answer'])

In [9]:
x_train.shape , x_test.shape


((1494, 14), (264, 14))

In [10]:
#function to build prompt and option pairs

def build_pairs(df):
    rows = []
    for i in df.index:
        prompt = df.loc[i, 'prompt_clean']
        options = ['A', 'B', 'C', 'D', 'E']
        for opt in options:
            text = prompt + ' ' + df.loc[i, f'{opt}_clean']
            label = 0
            if df.loc[i, 'answer'] == opt:
                label = 1

            rows.append({'id': df.loc[i, 'id'],
                         'option': opt,
                         'text': text,
                         'label': label})
    return pd.DataFrame(rows)

train_pairs = build_pairs(x_train)
val_pairs = build_pairs(x_test)

print(train_pairs.shape, val_pairs.shape)

(7470, 4) (1320, 4)


In [11]:
train_pairs.iloc[0:5]

,id,option,text,label
0,437,A,determine the correct option what is bollard p...,0
1,437,B,determine the correct option what is bollard p...,0
2,437,C,determine the correct option what is bollard p...,0
3,437,D,determine the correct option what is bollard p...,1
4,437,E,determine the correct option what is bollard p...,0


In [12]:
from collections import Counter

def build_vocab(texts, max_vocab_size=10000, min_freq=1):
    counter = Counter()
    for text in texts:
        tokens = text.split()
        counter.update(tokens)
    
    most_common = counter.most_common(max_vocab_size)
    
    word2idx = {'<PAD>': 0, '<UNK>': 1}
    for word,freq in most_common:
        if freq >= min_freq:
            word2idx[word] = len(word2idx)
    
    return word2idx

word2idx = build_vocab(train_pairs['text'], max_vocab_size=10000)
print(f"Vocab size: {len(word2idx)}")

Vocab size: 3084


In [13]:
token_lengths = train_pairs['text'].apply(lambda x: len(x.split()))
print(token_lengths.describe())
print("95th percentile:", token_lengths.quantile(0.95))

count    7470.000000
mean       44.142169
std        18.648359
min         9.000000
25%        31.000000
50%        40.000000
75%        55.000000
max       148.000000
Name: text, dtype: float64
95th percentile: 81.0
